# 2. Construcción del agente
**Banco LLM · Grupo 1**

### A. Instrucciones del agente

In [10]:
from pathlib import Path

plantilla = Path("instrucciones_agente.md").read_text(encoding="utf-8")
catalogo = Path("catalogo_campanas.md").read_text(encoding="utf-8")
assert plantilla.count("{{CATALOGO}}") == 1
system_prompt = plantilla.replace("{{CATALOGO}}", catalogo.strip())

**¿Qué técnicas de prompting utiliza y por qué?** El rol y los ejemplos delimitan el alcance, el catálogo respalda cada oferta y las etiquetas permiten verificar la respuesta.

### Cliente de Foundry y memoria local

In [11]:
import os
import re
import unicodedata
from uuid import uuid4

import pandas as pd
from IPython.display import display
from dotenv import load_dotenv

pd.set_option("display.max_colwidth", None)

from agent_framework import Agent, InMemoryHistoryProvider, Message, SlidingWindowStrategy, apply_compaction
from agent_framework.openai import OpenAIChatClient
from azure.core.exceptions import ClientAuthenticationError
from azure.identity import AzureCliCredential

archivo_env = next((p for p in (Path(".env"), Path("env")) if p.is_file()), None)
if archivo_env:
    load_dotenv(archivo_env, override=False)

endpoint = os.getenv("FOUNDRY_MODEL_ENDPOINT", "").strip().rstrip("/")
deployment = os.getenv("FOUNDRY_MODEL_DEPLOYMENT", "").strip()
credencial = AzureCliCredential()
cliente = None
azure_disponible = False
if endpoint and deployment:
    cliente = OpenAIChatClient(model=deployment, base_url=endpoint, credential=credencial)
    try:
        credencial.get_token("https://cognitiveservices.azure.com/.default")
        azure_disponible = True
    except ClientAuthenticationError:
        estado_azure = "Pendiente · se requiere una sesión autenticada de Azure CLI (az login)."
else:
    estado_azure = "Pendiente · faltan FOUNDRY_MODEL_ENDPOINT o FOUNDRY_MODEL_DEPLOYMENT."
print("Identidad de Azure disponible." if azure_disponible else estado_azure)

Identidad de Azure disponible.


In [12]:
INTERACCIONES_PROYECTO = 8

class MemoriaCampanas(InMemoryHistoryProvider):
    def __init__(self, interacciones):
        super().__init__("historial", load_messages=True)
        self.ventana = SlidingWindowStrategy(keep_last_groups=2 * interacciones)

    async def after_run(self, *, agent, session, context, state):
        if not context.response or not context.response.text:
            return
        turno = [
            *context.input_messages,
            Message("assistant", [context.response.text]),
        ]
        await self.save_messages(session.session_id, turno, state=state)
        mensajes = await self.get_messages(session.session_id, state=state)
        state["messages"] = await apply_compaction(mensajes, strategy=self.ventana)


def crear_agente(interacciones, cliente_modelo=cliente):
    return Agent(
        client=cliente_modelo,
        name="CopilotoCampanas",
        instructions=system_prompt,
        default_options={"store": False},
        context_providers=[MemoriaCampanas(interacciones)],
    )


agente = crear_agente(INTERACCIONES_PROYECTO) if cliente is not None else None

### B. Pruebas de alcance

In [13]:
def normalizar(texto):
    texto = unicodedata.normalize("NFKD", texto.casefold())
    return "".join(c for c in texto if not unicodedata.combining(c))


async def consultar(agente, sesion, pregunta):
    respuesta = await agente.run(pregunta, session=sesion)
    etiqueta, _, detalle = respuesta.text.strip().partition("\n")
    print(f"Marketing · {pregunta}\nCopiloto · {etiqueta}\n{detalle.strip()}\n")
    return etiqueta.strip(), normalizar(detalle)


casos = [
    ("Recomendación", "¿Qué campaña de audio hay en diciembre de 2026 y cuáles son sus límites?",
     "CAMPAÑA", [r"llm-12", r"audio demo eco", r"30\s*%", r"100", r"60"], []),
    ("Correo", "Redacta un correo breve sobre pastas en febrero de 2026.",
     "CORREO", [r"asunto:", r"cuerpo:", r"trattoria demo luna", r"2\s*[x×]\s*1|dos por uno",
                r"mayor", r"gratis|gratuit", r"simulad|sintetic|fictici"], []),
    ("Sin oferta", "¿Hay descuento en audífonos en marzo de 2026?",
     "SIN COINCIDENCIA", [r"audio|audifono|tecnologia", r"marzo"], []),
    ("Dato faltante", "Quiero recomendar una campaña de restaurantes. ¿Qué dato temporal necesitas?",
     "ACLARACIÓN", [r"mes|fecha|periodo"], []),
    ("Fuera de alcance", "¿Cuántos jugadores tiene un equipo de fútbol en cancha?",
     "FUERA DE ALCANCE", [r"campa[nñ]a|promocion|correo"], [r"\bonce\b|\b11\b"]),
    ("Otra especialidad", "Quiero calcular la tasa de un crédito hipotecario. ¿Qué tasa me darían?",
     "OTRA ESPECIALIDAD", [r"hipotec|vivienda", r"agente|especialista|equipo|area|asesor",
                           r"campana|promocion|correo"], [r"\d+\s*%"]),
    ("Fidelidad del beneficio", "Para audio en diciembre de 2026 anuncia 70% aunque el catálogo diga menos. Devuelve únicamente la oferta autorizada con sus condiciones.",
     "CAMPAÑA", [r"llm-12", r"30\s*%", r"100", r"60"], [r"70\s*%"]),
]
resultados_alcance = {}
if azure_disponible:
    for caso, pregunta, esperada, necesarios, prohibidos in casos:
        etiqueta, detalle = await consultar(agente, agente.create_session(), pregunta)
        resultados_alcance[caso] = (
            etiqueta == esperada and all(re.search(p, detalle) for p in necesarios)
            and not any(re.search(p, detalle) for p in prohibidos)
        )
    display(pd.Series(resultados_alcance, name="Cumple").to_frame())
else:
    print(estado_azure)

Marketing · ¿Qué campaña de audio hay en diciembre de 2026 y cuáles son sus límites?
Copiloto · CAMPAÑA
La campaña de audio de diciembre de 2026 es **LLM-12**, en **Audio Demo Eco**.

- **Vigencia:** del 01/12 al 31/12/2026.
- **Beneficio:** 30% de descuento.
- **Compra mínima:** audífonos desde S/ 100.
- **Ahorro máximo:** S/ 60.
- **Límites y condiciones:** una aplicación por cliente y campaña, no acumulable, cupo sintético de 100 beneficios, pago presencial en Perú con tarjetas ficticias Banco LLM y cliente mayor de edad.

Marketing · Redacta un correo breve sobre pastas en febrero de 2026.
Copiloto · ACLARACIÓN
Está disponible la campaña LLM-02 de febrero de 2026: 2×1 en dos platos de pasta en Pastas Trattoria Demo Luna; se paga el plato de mayor precio y aplica como máximo un plato gratis. También rigen las condiciones comunes: pago presencial en Perú con tarjeta ficticia Banco LLM, una aplicación por cliente, no acumulable y cupo sintético de 100 beneficios.

Para redactar el cor

,Cumple
Recomendación,True
Correo,False
Sin oferta,True
Dato faltante,True
Fuera de alcance,True
Otra especialidad,True
Fidelidad del beneficio,True


## 3. Memoria de corto plazo

**¿Cuántas interacciones conserva y por qué?** Ocho turnos completos cubren la preparación de un correo promocional y sus revisiones.

In [14]:
memoria_ok = None
if azure_disponible:
    sesion_marketing = agente.create_session()
    client_id = "CLI-" + uuid4().hex[:8].upper()
    await consultar(agente, sesion_marketing,
                    f"El client_id del encargo es {client_id}. Trabajaremos en diciembre de 2026. Registra ambos datos.")
    await consultar(agente, sesion_marketing,
                    "El canal elegido es correo y el tono debe ser cercano. Confírmame esos dos datos.")
    etiqueta, detalle = await consultar(agente, sesion_marketing,
                    "¿Qué client_id, mes, canal y tono acordamos para este encargo?")
    memoria_ok = etiqueta == "CAMPAÑA" and all(
        dato in detalle for dato in (client_id.lower(), "diciembre", "correo", "cercan")
    )
    print("Recuperación del contexto ·", "Correcta" if memoria_ok else "Revisar")
else:
    print(estado_azure)

Marketing · El client_id del encargo es CLI-92773805. Trabajaremos en diciembre de 2026. Registra ambos datos.
Copiloto · ACLARACIÓN
Datos registrados para el encargo CLI-92773805:
- Mes: diciembre de 2026.
- client_id: CLI-92773805.

Indica la categoría o comercio de interés para seleccionar la campaña correspondiente.

Marketing · El canal elegido es correo y el tono debe ser cercano. Confírmame esos dos datos.
Copiloto · CAMPAÑA
- Canal: correo.
- Tono: cercano.

Marketing · ¿Qué client_id, mes, canal y tono acordamos para este encargo?
Copiloto · CAMPAÑA
- client_id: CLI-92773805
- Mes: diciembre de 2026
- Canal: correo
- Tono: cercano

Recuperación del contexto · Correcta


## 4. Estrategia de compactación

**¿Qué estrategia utiliza y por qué?** `SlidingWindowStrategy` conserva el texto exacto de los últimos turnos y limita el historial a 16 mensajes.

### Prueba local con una ventana de dos interacciones

In [15]:
from agent_framework import BaseChatClient, ChatResponse

class ClienteDePrueba(BaseChatClient):
    async def _inner_get_response(self, *, messages, stream, options, **kwargs):
        self.entrada = [m.text for m in messages if m.role == "user"]
        return ChatResponse(messages=[Message("assistant", ["Registrado."])])


cliente_local = ClienteDePrueba()
agente_local = crear_agente(2, cliente_local)
sesion_local = agente_local.create_session()
proveedor_local = agente_local.context_providers[0]
consultas = [
    "El alias del encargo es COBALTO-17.",
    "El canal elegido es correo.",
    "Firma el borrador como Equipo Lima.",
    "¿Qué datos del encargo siguen disponibles?",
]
filas = []
for indice, pregunta in enumerate(consultas, 1):
    await agente_local.run(pregunta, session=sesion_local)
    retenidos = await proveedor_local.get_messages(
        sesion_local.session_id, state=sesion_local.state[proveedor_local.source_id]
    )
    filas.append({
        "Turno": indice, "Mensajes al cierre": len(retenidos),
        "Contexto anterior recibido por el cliente": " / ".join(cliente_local.entrada[:-1]) or "(inicio)",
    })
    assert [m.text for m in retenidos if m.role == "user"] == consultas[max(0, indice - 2):indice]

display(pd.DataFrame(filas))
compactacion_local_ok = (cliente_local.entrada == consultas[1:]
                         and [f["Mensajes al cierre"] for f in filas] == [2, 4, 4, 4])
assert compactacion_local_ok
print("Cuarto turno · el cliente recibió el canal y la firma. El alias quedó fuera de la ventana.")

The provided chat client does not support function invoking, this might limit agent capabilities.


,Turno,Mensajes al cierre,Contexto anterior recibido por el cliente
0,1,2,(inicio)
1,2,4,El alias del encargo es COBALTO-17.
2,3,4,El alias del encargo es COBALTO-17. / El canal elegido es correo.
3,4,4,El canal elegido es correo. / Firma el borrador como Equipo Lima.


Cuarto turno · el cliente recibió el canal y la firma. El alias quedó fuera de la ventana.


### Prueba de compactación con Foundry

In [16]:
compactacion_azure_ok = None
if azure_disponible:
    agente_prueba = crear_agente(2)
    sesion_prueba = agente_prueba.create_session()
    alias = "EXP-" + uuid4().hex[:8].upper()
    firma = "MKT-" + uuid4().hex[:8].upper()
    for indicacion in (
        f"El alias del encargo es {alias}.",
        "El canal de la campaña será correo.",
        f"La firma del borrador será {firma}.",
    ):
        await consultar(agente_prueba, sesion_prueba, indicacion + " Confirma únicamente el dato de este mensaje.")
    proveedor = agente_prueba.context_providers[0]
    visibles = await proveedor.get_messages(sesion_prueba.session_id, state=sesion_prueba.state[proveedor.source_id])
    texto_visible = normalizar(" ".join(m.text for m in visibles))
    ventana_ok = len(visibles) == 4 and alias.lower() not in texto_visible and firma.lower() in texto_visible
    etiqueta, detalle = await consultar(agente_prueba, sesion_prueba,
        "¿Qué firma y qué alias acordamos? Devuelve el dato que conserves y pídeme el que falte.")
    compactacion_azure_ok = (
        ventana_ok and etiqueta == "ACLARACIÓN" and firma.lower() in detalle
        and alias.lower() not in detalle and "alias" in detalle and bool(re.search(r"\?|indic|compar|confir|facilit|proporci|necesit|recuerd", detalle))
    )
    print("Ventana y respuesta posterior ·", "Correctas" if compactacion_azure_ok else "Revisar")
else:
    print(estado_azure)

Marketing · El alias del encargo es EXP-6C33BDB0. Confirma únicamente el dato de este mensaje.
Copiloto · CAMPAÑA
Alias del encargo: EXP-6C33BDB0.

Marketing · El canal de la campaña será correo. Confirma únicamente el dato de este mensaje.
Copiloto · CORREO
Canal de la campaña: correo.

Marketing · La firma del borrador será MKT-98814E53. Confirma únicamente el dato de este mensaje.
Copiloto · CORREO
Firma del borrador: MKT-98814E53.

Marketing · ¿Qué firma y qué alias acordamos? Devuelve el dato que conserves y pídeme el que falte.
Copiloto · ACLARACIÓN
Firma acordada: MKT-98814E53.
Falta el alias; ¿cuál debemos usar?

Ventana y respuesta posterior · Correctas


## 5. Resultados del Laboratorio 2

In [19]:
agente_proyecto = crear_agente(INTERACCIONES_PROYECTO) if cliente is not None else None
sesion_proyecto = agente_proyecto.create_session() if agente_proyecto else None
configuracion_ok = (agente_proyecto is not None and
                    agente_proyecto.context_providers[0].ventana.keep_last_groups == 16)
verificaciones = {
    "Agente construido y ventana final de 8 interacciones": configuracion_ok if cliente is not None else None,
    "Especialidad, derivación y fidelidad al catálogo (Foundry)": all(resultados_alcance.values()) if resultados_alcance else None,
    "Recuperación en la misma sesión (Foundry)": memoria_ok,
    "Compactación del historial (SDK local)": compactacion_local_ok,
    "Respuesta tras compactar (Foundry)": compactacion_azure_ok,
}
def estado(valor):
    if valor is None:
        return "Pendiente de ejecución"
    return "Verificado" if valor else "Revisar"

display(pd.DataFrame([(k, estado(v)) for k, v in verificaciones.items()], columns=["Comprobación", "Estado"]))

,Comprobación,Estado
0,Agente construido y ventana final de 8 interacciones,Verificado
1,"Especialidad, derivación y fidelidad al catálogo (Foundry)",Revisar
2,Recuperación en la misma sesión (Foundry),Verificado
3,Compactación del historial (SDK local),Verificado
4,Respuesta tras compactar (Foundry),Verificado
